# İlişki denemesi · Öğretmen "konuşan hangi kararı istiyor?" sorusunu tutarlı cevaplıyor mu?
Elimizdeki **ham kartlardan** saray içi karakterlerin kartlarını alır. Gemma 4 12B'ye her kart için **iki kez** sorar: "Konuşan hangi kararı istiyor? Sol / Sağ / Belirsiz". Sonuçta üç sayı çıkar:
- **Tutarlılık:** İki cevap kaç kartta aynı?
- **Belirsiz oranı:** Model kaç kartta karar veremedi?
- **Elle kontrol:** Ekrana basılan 40 karta siz bakıp doğru mu diye işaretlersiniz.

**Karar ölçütü:** Tutarlılık %90 veya üstü, Belirsiz %10 veya altı, 40 kartın en az 36'sı doğruysa `İstediği:` satırını kart biçimine ekleriz.

**Colab:** T4 GPU → Tümünü çalıştır. Kart dosyaları Drive'da `kaydir_veri/` altında olmalı.
**Kaggle:** GPU T4 x2 + Internet On + `HF_TOKEN` secret. `kaydir_veri.zip`'i *Add Input → Upload* ile veri seti olarak ekle.

In [ ]:
SARAY = ['Vezir', 'Kadı', 'Başkâtip', 'Saray Hatun', 'Beylerbeyi', 'Hazinedar', 'Soytarı', 'Elçi', 'Kale Kumandanı']
EN_FAZLA = 600            # en fazla kaç kart etiketlensin (600 kart ≈ 15-25 dk)
import os
KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
GIRDI = '/kaggle/input/**/kart_*.jsonl' if KAGGLE else '/content/drive/MyDrive/kaydir_veri/**/kart_*.jsonl'

In [ ]:
import sys, subprocess, glob, json, random, getpass
if not KAGGLE:
    from google.colab import drive; drive.mount('/content/drive')
for depo in ('AI', 'Docs'):
    if not os.path.isdir(f'/tmp/{depo}'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', f'https://github.com/IleriProgramlamaProje/{depo}', f'/tmp/{depo}'], check=True)
sys.path.insert(0, '/tmp/AI')
if 'motor' not in globals():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'bitsandbytes'], check=True)
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret('HF_TOKEN')
    else:
        from google.colab import userdata; token = userdata.get('HF_TOKEN')
except Exception:
    token = None
from huggingface_hub import login
login(token=(token or getpass.getpass('Hugging Face token: ')).strip())

In [ ]:
kartlar, gorulen = [], set()
for yol in sorted(glob.glob(GIRDI, recursive=True)):
    for satir in open(yol, encoding='utf-8'):
        try:
            k = json.loads(satir)
        except ValueError:
            continue
        m, metin = k.get('malzeme', {}), (k.get('cikti') or '').strip()
        if m.get('karakter') in SARAY and metin.startswith('Metin:') and 'Sağ etki' in metin and metin not in gorulen:
            gorulen.add(metin); kartlar.append({'karakter': m['karakter'], 'kart': metin, 'dosya': os.path.basename(yol)})
random.Random(1).shuffle(kartlar)
kartlar = kartlar[:EN_FAZLA]
from collections import Counter
print(len(kartlar), 'kart ·', dict(Counter(k['karakter'] for k in kartlar)))
assert kartlar, 'Kart bulunamadı: GIRDI yolunu kontrol et'

In [ ]:
import torch
from ai.veri_uretimi.uretici import motor_ac
tema = json.load(open('/tmp/Docs/tema.json', encoding='utf-8'))
GOREV = {k['unvan']: k.get('gorev', '') for k in tema['karakterler']}
if 'motor' not in globals():
    motor, _ = motor_ac('transformers8bit' if torch.cuda.device_count() >= 2 else 'transformers4bit', 'google/gemma-4-12B-it')
SORU = '''Aşağıda bir kart oyunundan bir kart var. Kartta {karakter} hükümdara bir dert anlatıyor ve iki karar sunuluyor (Sol ve Sağ).
{karakter}'in görevi: {gorev}

{kart}

Soru: {karakter}, kendi derdine ve görevine göre hükümdarın hangi kararı vermesini ister?
Sadece tek kelime yaz: Sol, Sağ ya da Belirsiz.'''
def sor(liste, tur):
    cevaplar = []
    for i in range(0, len(liste), 8):
        grup = liste[i:i + 8]
        for (metin, _), k in zip(motor.uret([SORU.format(karakter=k['karakter'], gorev=GOREV.get(k['karakter'], ''), kart=k['kart']) for k in grup]), grup):
            ilk = (metin.strip().split() or [''])[0].strip('.:*').lower()
            cevaplar.append('Sol' if ilk.startswith('sol') else 'Sağ' if ilk.startswith(('sağ', 'sag')) else 'Belirsiz')
        print(f'{tur}. tur: {min(i + 8, len(liste))}/{len(liste)}', end='\r')
    print()
    return cevaplar
c1 = sor(kartlar, 1); c2 = sor(kartlar, 2)

In [ ]:
for k, a, b in zip(kartlar, c1, c2):
    k['istedigi_1'], k['istedigi_2'] = a, b
n = len(kartlar)
ayni = sum(a == b for a, b in zip(c1, c2))
belirsiz = sum('Belirsiz' in (a, b) for a, b in zip(c1, c2))
print(f'Tutarlılık: {ayni}/{n} = %{100 * ayni / n:.0f}   (hedef ≥ %90)')
print(f'Belirsiz:   {belirsiz}/{n} = %{100 * belirsiz / n:.0f}   (hedef ≤ %10)')
print('Dağılım (1. tur):', dict(Counter(c1)))
print('Karakter bazında tutarlılık:')
for kar in SARAY:
    s = [(a, b) for k, a, b in zip(kartlar, c1, c2) if k['karakter'] == kar]
    if s: print(f'  {kar}: %{100 * sum(a == b for a, b in s) / len(s):.0f} ({len(s)} kart)')
with open('iliski_deneme.jsonl', 'w', encoding='utf-8') as f:
    for k in kartlar: f.write(json.dumps(k, ensure_ascii=False) + '\n')
print('Kaydedildi: iliski_deneme.jsonl')

## Elle kontrol: 40 kart
Her kartı oku. Konuşan gerçekten o kararı mı ister? Doğru sayısını not et (hedef 40'ta en az 36).

In [ ]:
for i, (k, a, b) in enumerate(zip(kartlar[:40], c1, c2), 1):
    print(f'===== {i} · {k["karakter"]} · model: {a} / {b}')
    print(k['kart'], '\n')

In [ ]:
if KAGGLE:
    print('Output sekmesinden iliski_deneme.jsonl dosyasını indir')
else:
    from google.colab import files; files.download('iliski_deneme.jsonl')